## PydanticAI 기본 실습

**실습 목표:**

1. **PydanticAI의 Agent 개념** 을 이해하고 기본 텍스트 생성을 수행할 수 있다
2. **매개변수(temperature, max_tokens 등)** 를 조절하여 AI 응답을 제어할 수 있다
3. **스트리밍 응답** 으로 AI 답변을 실시간으로 출력할 수 있다
4. **멀티턴 대화** 를 구현하여 대화 맥락을 유지할 수 있다
5. **도구(Tool)** 를 등록하여 Agent의 기능을 확장할 수 있다
6. **비동기 동시 호출** 로 여러 요청을 병렬 처리할 수 있다

**사용 프레임워크:** [PydanticAI](https://ai.pydantic.dev/) — Python 에이전트 프레임워크

### AI Agent란?

**AI Agent** 는 단순히 질문에 답하는 챗봇을 넘어서, **스스로 판단하고 도구를 사용하여 작업을 수행하는 AI 시스템** 입니다.

![AI Agent는 지우와 피카츄의 관계와 같습니다](images/ai_agent_pokemon.png)

**포켓몬으로 이해하는 AI Agent**

| 포켓몬 세계 | AI Agent 세계 | 설명 |
|------------|-------------|------|
| **지우** (트레이너) | **개발자** (나) | 목표를 정하고 지시를 내리는 사람 |
| **피카츄** (포켓몬) | **Agent** (LLM + 도구 + 지침) | 지시를 받고 스스로 판단하여 행동하는 존재 |
| **기술** (10만볼트, 전광석화) | **도구 (Tool)** | Agent가 상황에 맞게 선택하여 호출하는 기능 |
| **지우의 전략/지시** | **시스템 프롬프트 (instruction)** | Agent가 어떻게 행동할지 정하는 지침 |

**일반 LLM 호출 vs AI Agent 비교:**

```
[일반 LLM 호출]  지우: "피카츄, 저기 뭐야?" => 피카츄: "피카" (대답만 함)

[AI Agent]       지우: "피카츄, 저 포켓몬 잡아!" 
                 => 피카츄가 스스로 판단: "날아다니니까 10만볼트!"  (도구 선택)
                 => 10만볼트 사용                                  (도구 실행)
                 => "잡았다! HP 35, 타입: 비행"                    (결과 반환)
```

| 구분 | 일반 LLM 호출 | AI Agent |
|------|-------------|----------|
| **동작 방식** | 질문 => 답변 (1회성) | 목표를 받고 => 판단 => 도구 사용 => 결과 반환 |
| **도구 사용** | 불가능 | 검색, 계산, API 호출 등 **외부 도구를 직접 호출** |
| **판단력** | 주어진 질문에만 답변 | 어떤 도구를 쓸지 **스스로 판단** |
| **상태 관리** | 이전 대화를 기억하지 못함 | 대화 맥락을 유지하며 **연속 작업 가능** |

> **이 실습에서는** PydanticAI 프레임워크를 사용하여 AI Agent를 만들고,
> 도구 등록, 구조화된 출력, 멀티턴 대화 등 Agent의 핵심 기능을 실습합니다.

### 환경 설정

실습에 필요한 패키지를 설치하고, API 키를 설정합니다.

In [ ]:
# 필요한 패키지를 한 번에 설치합니다
# !pip install -q pydantic-ai-slim[google] python-dotenv pydantic

# uv 권장! uv 환경에서는 위 pip 대신 터미널에서 uv sync 명령어로 설치하세요:
# uv sync

**PydanticAI 모델별 설정 가이드**

PydanticAI는 다양한 LLM 프로바이더를 지원합니다. 각 프로바이더별 **모델 ID 형식** 과 **환경변수** 설정 방법은 아래와 같습니다.

| 프로바이더 | 모델 ID 형식 | 모델 예시 | 환경변수 (API Key) | 설치 패키지 |
|---|---|---|---|---|
| **Google Gemini** | `google-gla:{모델명}` | `google-gla:gemini-2.5-flash` | `GEMINI_API_KEY` | `pydantic-ai-slim[google]` |
| **OpenAI** | `openai:{모델명}` | `openai:gpt-4o`, `openai:gpt-4o-mini` | `OPENAI_API_KEY` | `pydantic-ai-slim[openai]` |
| **Anthropic (Claude)** | `anthropic:{모델명}` | `anthropic:claude-sonnet-4-20250514` | `ANTHROPIC_API_KEY` | `pydantic-ai-slim[anthropic]` |

**`.env` 파일 설정 예시:**

```
# Google Gemini
GEMINI_API_KEY=your-gemini-api-key

# OpenAI
OPENAI_API_KEY=your-openai-api-key

# Anthropic (Claude)
ANTHROPIC_API_KEY=your-anthropic-api-key
```

**코드 사용 예시:**

```python
from pydantic_ai import Agent

# Google Gemini
agent = Agent("google-gla:gemini-2.5-flash")

# OpenAI
agent = Agent("openai:gpt-4o")

# Anthropic Claude
agent = Agent("anthropic:claude-sonnet-4-20250514")
```

**공식 문서 링크:**

- [PydanticAI 모델 설정 개요](https://ai.pydantic.dev/models/)
- [PydanticAI - Google Gemini 설정](https://ai.pydantic.dev/models/google/)
- [PydanticAI - OpenAI 설정](https://ai.pydantic.dev/models/openai/)
- [PydanticAI - Anthropic (Claude) 설정](https://ai.pydantic.dev/models/anthropic/)

**실습**
- 환경변수를 로드하고 API 키가 올바르게 설정되었는지 확인합니다.
- `.env` 파일에 `GEMINI_API_KEY`를 설정한 후 아래 셀을 실행합니다.

In [1]:
# ========================================
# 필요한 라이브러리를 불러옵니다
# ========================================

import os
from pprint import pprint
from dotenv import load_dotenv

# .env 파일에서 API 키 로드
load_dotenv()
api_key = os.getenv('GEMINI_API_KEY', '.env에 key가 없는 경우 기본값')
gemini_model = os.getenv('GEMINI_MODEL', 'gemini-3.1-flash-lite-preview')

# PydanticAI는 GEMINI_API_KEY 환경변수를 자동으로 인식합니다
# 모델 ID 형식: 'google-gla:{모델명}'
model_id = f'google-gla:{gemini_model}'

# API 키 유효성 검사
api_key_valid = api_key and 'YOUR_API_KEY' not in api_key
print(f"API 키 설정 확인: {'✓' if api_key_valid else '✗'}")
if not api_key_valid:
    print("⚠️  .env 파일에서 GEMINI_API_KEY를 실제 API 키로 설정해주세요!")
print(f"모델 확인: {model_id}")

API 키 설정 확인: ✓
모델 확인: google-gla:gemini-3.1-flash-lite-preview


### PydanticAI란?

- [PydanticAI 공식 문서](https://ai.pydantic.dev/)

PydanticAI는 **생성형 AI 애플리케이션을 쉽게 만들기 위한 Python 에이전트 프레임워크** 입니다.

웹 개발에서 FastAPI가 API 서버를 쉽게 만들어주는 것처럼, PydanticAI는 AI 에이전트를 쉽게 만들어줍니다.

**핵심 개념:**

| 개념 | 설명 | 비유 |
|------|------|------|
| **Agent** | AI와 대화하는 단위. 모델, 시스템 프롬프트, 도구를 하나로 묶음 | 특정 역할을 하는 직원 |
| **Structured Output** | AI 응답을 Pydantic 모델(스키마)로 강제 | 정해진 양식에 맞춰 보고서 작성 |
| **Tool** | Agent가 호출할 수 있는 Python 함수 | 직원이 사용하는 도구(계산기, DB 등) |
| **Dependency Injection** | 실행 시점에 필요한 데이터를 주입 | 직원에게 업무에 필요한 자료 전달 |

### 1. 기본 텍스트 생성 (Single-turn)

- [Agent 공식 문서](https://ai.pydantic.dev/agents/)

**(1) Agent와 실행 방식**

PydanticAI에서 AI와 대화하려면 먼저 **Agent** 를 만들어야 합니다.

```python
agent = Agent('google-gla:gemini-3.1-flash-lite-preview')  # Agent 생성 (모델 지정)
result = await agent.run('질문 내용')                        # 비동기 실행
print(result.output)                                        # 응답 텍스트
```

**실행 방식 3가지:**

| 메서드 | 방식 | 사용 상황 |
|--------|------|----------|
| `agent.run_sync()` | 동기(blocking) | 일반 Python 스크립트 (이벤트 루프 없을 때) |
| `await agent.run()` | 비동기(async) | 동시 호출 시, Jupyter 노트북 |
| `await agent.run_stream()` | 비동기 스트리밍 | 실시간 응답 출력 |

> **참고:** Jupyter 노트북은 내부적으로 이벤트 루프가 이미 실행 중이므로 `run_sync()`를 사용하면 충돌이 발생합니다. Jupyter에서는 반드시 `await agent.run()`을 사용합니다.

**실습 (1)**
- Agent를 생성하고 기본 질문을 보내봅니다.
- `Agent(model_id)`로 Agent를 만들고, `await agent.run()`으로 질문합니다.

In [2]:
from pydantic_ai import Agent

# Agent 생성 - 모델만 지정하면 바로 사용 가능
agent = Agent(model_id) # 모델 ID는 어디서 확인??? gemini 공식문서에 모델 ID

# 기본 텍스트 생성 요청 (비동기 방식 — Jupyter에서는 await 사용)
search_query = "스파르타 부트캠프 데이터 분석 트랙에 대해서 알려줘"

result = await agent.run(search_query) # 비동기 함수의 작업이 완료까지 기다린다!

# 응답 출력
print(f"질문: {search_query}")
print(f"답변: {result.output}")
print("-" * 50)

질문: 스파르타 부트캠프 데이터 분석 트랙에 대해서 알려줘
답변: 스파르타코딩클럽의 **데이터 분석 부트캠프(데이터 분석 트랙)**는 주로 비전공자나 데이터 분석 직무로 커리어 전환을 희망하는 분들을 대상으로 하는 실무 중심의 교육 과정입니다.

전반적인 특징과 장단점을 정리해 드립니다. (과정명은 기수나 프로모션에 따라 '데이터 분석가 취업 캠프' 등으로 조금씩 달라질 수 있습니다.)

---

### 1. 주요 특징 및 커리큘럼
스파르타의 교육 철학인 **'결과물을 만들어내는 것'**에 집중되어 있습니다.

*   **언어 및 도구:** 파이썬(Python), SQL, Pandas, Matplotlib/Seaborn(시각화), Tableau/Power BI 등 실무에서 가장 많이 쓰이는 툴을 중심으로 학습합니다.
*   **실무 프로젝트:** 단순 이론 학습이 아니라, 실제 공공 데이터나 기업 데이터를 활용해 대시보드를 만들거나 분석 보고서를 작성하는 프로젝트 비중이 높습니다.
*   **튜터링 시스템:** 1:1 질의응답 및 코드 리뷰를 지원하며, 현업 데이터 분석가 출신 튜터들이 학습을 돕습니다.
*   **커리어 관리:** 이력서 첨삭, 모의 면접, 자소서 코칭 등 취업 지원 프로그램이 포함되어 있습니다.

### 2. 장점
*   **비전공자 친화적:** 코딩을 처음 접하는 사람도 따라올 수 있도록 기초부터 탄탄하게 다지는 커리큘럼을 가지고 있습니다.
*   **실무 중심(Hands-on):** 이론보다 "그래서 이걸로 어떤 문제를 해결할 수 있는가?"에 초점을 맞추어 실무 환경과 유사한 경험을 제공합니다.
*   **커뮤니티:** 수강생들 간의 네트워킹과 분위기 관리가 잘 되어 있어, 혼자 독학할 때보다 끝까지 완주할 확률이 높습니다.
*   **SQL 강조:** 데이터 분석가 채용 시 가장 중요한 스킬인 SQL 활용 능력을 집중적으로 훈련합니다.

### 3. 고려해야 할 점 (단점 및 주의사항)
*   **학습 강도:** 부트캠프 특성상 학습량

HTTP 응답코드
200번 : 성공
400번 : 내가 잘못 요청
500번 : 서버의 오류

**실습 (2)**
- 응답 객체의 내부 구조를 확인합니다.
- `result.output`, `result.usage()`, `result.new_messages()`를 살펴봅니다.

In [3]:
result
type(result) #pydantic_ai.run.AgentRunResult

pydantic_ai.run.AgentRunResult

In [4]:
# 응답 객체를 더 자세히 살펴봅시다
print("=" * 50)
print("[응답 객체 구조]")
print("=" * 50)

# 1. 응답 텍스트
print(f"\n[output] 응답 텍스트:")
print(result.output[:200], "...")

# 2. 토큰 사용량
print(f"\n[usage] 토큰 사용량:")
pprint(result.usage())
# RunUsage(input_tokens=17, output_tokens=897, details={'text_prompt_tokens': 17}, requests=1)
# 비용 계산!

# 3. 대화 메시지 기록 (멀티턴에서 활용)
print(f"\n[new_messages] 대화 기록: {len(result.new_messages())}개")
for msg in result.new_messages():
    print(f"  - {type(msg).__name__}: {str(msg)[:100]}...")

[응답 객체 구조]

[output] 응답 텍스트:
스파르타코딩클럽의 **데이터 분석 부트캠프(데이터 분석 트랙)**는 주로 비전공자나 데이터 분석 직무로 커리어 전환을 희망하는 분들을 대상으로 하는 실무 중심의 교육 과정입니다.

전반적인 특징과 장단점을 정리해 드립니다. (과정명은 기수나 프로모션에 따라 '데이터 분석가 취업 캠프' 등으로 조금씩 달라질 수 있습니다.)

---

### 1. 주요 특징  ...

[usage] 토큰 사용량:
RunUsage(input_tokens=17, output_tokens=983, details={'text_prompt_tokens': 17}, requests=1)

[new_messages] 대화 기록: 2개
  - ModelRequest: ModelRequest(parts=[UserPromptPart(content='스파르타 부트캠프 데이터 분석 트랙에 대해서 알려줘', timestamp=datetime.dateti...
  - ModelResponse: ModelResponse(parts=[TextPart(content='스파르타코딩클럽의 **데이터 분석 부트캠프(데이터 분석 트랙)**는 주로 비전공자나 데이터 분석 직무로 커리어...


**(2) 토큰 사용량 확인 및 비용 산출**

- [Gemini 모델 비용 문서 링크](https://ai.google.dev/gemini-api/docs/pricing?hl=ko)

PydanticAI에서는 `result.usage()` 메서드로 토큰 사용량을 확인합니다.

```python
usage = result.usage()
usage.input_tokens    # 입력 토큰 수
usage.output_tokens   # 출력 토큰 수
```

**실습 (3)**
- 위 실행 결과의 토큰 사용량과 비용을 계산합니다.
- Gemini 모델의 토큰당 가격을 기준으로 USD/KRW 비용을 산출합니다.

In [6]:
# 토큰 사용량 정보
usage = result.usage()
print(usage)

prompt_tokens = usage.input_tokens or 0     # 입력 토큰 수
output_tokens = usage.output_tokens or 0    # 응답 토큰 수

# gemini-3.1-flash-lite 가격 (2026-03-23 기준, 백만 토큰당 USD)
# 모델의 공식 문서 Gemini 공식문서 참고!
input_price = 0.25 #백만 토큰 당 가격
output_price = 0.50 #백만 토큰 당 가격
exchange_rate = 1500  # 환율

# 비용 계산 (USD)
input_cost_usd = (prompt_tokens / 1_000_000) * input_price
output_cost_usd = (output_tokens / 1_000_000) * output_price
total_cost_usd = input_cost_usd + output_cost_usd

# 비용 계산 (KRW)
total_cost_krw = total_cost_usd * exchange_rate

print(f"\n" + "=" * 60)
print(f"{'토큰 사용량 및 비용':^52}")
print("=" * 60)
print(f"프롬프트 토큰: {prompt_tokens:>6,} tokens")
print(f"              ${input_cost_usd:>9.6f}  (₩{input_cost_usd * exchange_rate:>7.4f})")
print(f"-" * 60)
print(f"응답 토큰:     {output_tokens:>6,} tokens")
print(f"              ${output_cost_usd:>9.6f}  (₩{output_cost_usd * exchange_rate:>7.4f})")
print(f"-" * 60)
print(f"총 비용:      ${total_cost_usd:>9.6f}  (₩{total_cost_krw:>7.4f})")
print("=" * 60)

RunUsage(input_tokens=17, output_tokens=983, details={'text_prompt_tokens': 17}, requests=1)

                    토큰 사용량 및 비용                     
프롬프트 토큰:     17 tokens
              $ 0.000004  (₩ 0.0064)
------------------------------------------------------------
응답 토큰:        983 tokens
              $ 0.000491  (₩ 0.7372)
------------------------------------------------------------
총 비용:      $ 0.000496  (₩ 0.7436)


# 최종 프로젝트에서 LLM API사용시 비용계산!
# 내 프로젝트에서 얼마나 비용이들지 !
# - 기업에서도 분석계획 <-- 비용도 넣어야 승인!

# 데이터 1000건을 LLM을 써야한다

# 약 10건에 대해서 실행해서 -> 1회 호출당 입력,출력 평균 토큰을 얻고
# 1회 호출시 비용 * 1000 ==> 1000회에 따른 대략적인 비용산출이 !!!

**(3) 매개변수를 활용한 상세 설정**

- [Pydantic AI - Google 모델 설정 문서](https://ai.pydantic.dev/models/google/)
- [Gemini API 가격 정책](https://ai.google.dev/gemini-api/docs/pricing?hl=ko)

PydanticAI에서는 `model_settings` 매개변수로 AI의 동작을 제어합니다.

![LLM 단어 생성 과정](images/LLM_단어생성.gif)

| 매개변수 | 설명 | 기본값 | 범위 |
|----------|------|--------|------|
| `temperature` | 창의성 조절 (높을수록 다양한 응답) | 1.0 | 0.0 ~ 2.0 |
| `max_tokens` | 최대 출력 토큰 수 | 모델별 상이 | 1 ~ 모델 한도 |
| `top_p` | 다양성 조절 (누적 확률 기반) | 0.95 | 0.0 ~ 1.0 |

**설정 방법 2가지:**

```python
# 방법 1: dict로 설정
result = await agent.run(query, model_settings={'temperature': 0.5, 'max_tokens': 200})

# 방법 2: GoogleModelSettings 객체 (Google 전용 설정 포함)
from pydantic_ai.models.google import GoogleModelSettings
settings = GoogleModelSettings(temperature=0.5, max_tokens=200)
result = await agent.run(query, model_settings=settings)
```

**Thinking 설정 (`google_thinking_config`)**

Gemini 모델의 **추론(thinking) 깊이** 를 조절할 수 있습니다. 모델 세대에 따라 설정 방식이 다릅니다.

| 모델 세대 | 설정 키 | 값 | 설명 |
|-----------|---------|-----|------|
| **Gemini 3** 이상 | `thinking_level` | `'low'`, `'high'` | 추론 깊이 수준 선택 |
| **Gemini 2.5** | `thinking_budget` | `0` ~ 정수값 | 추론 토큰 예산 (0 = 비활성화) |

```python
# Gemini 3: thinking_level로 추론 깊이 조절
settings = GoogleModelSettings(
    google_thinking_config={'thinking_level': 'low'}
)

# Gemini 2.5: thinking_budget으로 추론 토큰 수 제한
settings = GoogleModelSettings(
    google_thinking_config={'thinking_budget': 0}  # 0이면 thinking 비활성화
)
```

**실습 (4)**
- 시스템 프롬프트와 매개변수를 설정하여 AI 응답을 제어합니다.
- `GoogleModelSettings`로 temperature, max_tokens, top_p를 조절합니다.
- [GoogleModelSettings 공식 문서](https://ai.pydantic.dev/models/google/) | [Gemini API 생성 파라미터 가이드](https://ai.google.dev/gemini-api/docs/text-generation?hl=ko#configure)

모델이 단어(토큰)1개를 출력할 때 

모든 단어에 대한 확률 매긴다

고양이 : 0.3
강아지 : 0.25
다람쥐 : 0.1
우주 : 0.00005

top_p 매개변수 누적확률값
단어를 확률에대해서 내림차순 해서 누적확률값을 구해서 그 범위내에있는 단어들만 후보 단어로 선출


top_p = 0.3  #고양이
top_p = 0.5  #고양이 
top_p = 0.6  #고양이(0.3), 강아지(0.25) <-- 후보단어 
top_p = 0.65 #고양이, 강아지, 다람쥐

temperature = 0이면 무조건 고양이를 최종단어로 선출!
temperature = 2이면 고양이와 강아지의 확률이 비슷한 상태로 랜덤하게 선출!

temperature는 후보단어들의 확률값을 조정!
temperature값이 0에 가까우면 확률이 높은것은 더 높게 설정! 낮은 것은 낮게 만듬!
temperature값이 2에 후보단어들의 확률을 비슷 비슷하게 만들어준다!





In [20]:
from pydantic_ai.models.google import GoogleModelSettings

# 시스템 프롬프트가 포함된 Agent 생성
expert_agent = Agent(
    model_id,
    system_prompt="너는 AI 기술을 쉽게 설명하는 전문가야. 초보자도 이해할 수 있게 설명해줘."
)

# 매개변수 상세 설정
settings = GoogleModelSettings(
    temperature=0,     # 낮은 창의성 => 일관된 응답
    max_tokens=1000,      # 최대 200 토큰
    top_p=0.9,           # 다양성 조절
)

# 질문 설정
question = """
피카츄와 라이츄의 차이에 대해서 알려줘
"""
result = await expert_agent.run(question, model_settings=settings)
try:
   
    print(f"질문: {question}")
    print(f"모델: {model_id}")
    print(f"설정: 창의성={settings['temperature']}, 다양성={settings['top_p']}")
    print("=" * 80)
    print(f"\n응답:")
    print(result.output)

except Exception as e:
    print(f"API 호출 오류: {e}")

질문: 
피카츄와 라이츄의 차이에 대해서 알려줘

모델: google-gla:gemini-3.1-flash-lite-preview
설정: 창의성=0, 다양성=0.9

응답:
반가워요! AI 기술을 설명하는 전문가로서, **피카츄와 라이츄의 관계를 'AI 모델의 진화 과정'**에 빗대어 아주 쉽게 설명해 드릴게요.

---

### 1. 피카츄: "기본 모델 (Base Model)"
피카츄는 포켓몬 세계에서 가장 유명한 **'기본형'**이죠? AI로 치면 **'기본 모델(Base Model)'**과 같아요.

*   **특징:** 작고 귀엽고, 어디든 데리고 다니기 편해요. 하지만 전기 공격의 파워는 적당한 수준이죠.
*   **AI 관점:** 우리가 흔히 쓰는 챗봇이나 간단한 이미지 생성 AI처럼, **범용적으로 쓰기 좋고 가벼운 모델**이라고 생각하면 돼요. 일상적인 대화나 간단한 업무를 처리하는 데 최적화되어 있죠.

### 2. 라이츄: "고성능 모델 (Advanced/Fine-tuned Model)"
피카츄에게 '천둥의 돌'을 쓰면 라이츄가 되죠? 이건 AI가 **'더 높은 성능을 내기 위해 업그레이드된 상태'**라고 볼 수 있어요.

*   **특징:** 피카츄보다 덩치도 크고, 전기 공격의 위력이 훨씬 강력해요. 하지만 그만큼 에너지를 많이 쓰고, 다루기가 조금 더 까다로울 수 있죠.
*   **AI 관점:** 특정 분야(예: 코딩, 의학, 법률 등)에서 **전문적인 지식을 더 많이 학습시킨 '고성능 모델'**이에요. 복잡한 문제를 해결하거나 훨씬 정교한 결과물을 만들어낼 수 있지만, 그만큼 구동하는 데 더 많은 컴퓨터 자원(전력, 메모리 등)이 필요해요.

---

### 요약하자면 이렇습니다!

| 구분 | 피카츄 (기본 모델) | 라이츄 (고성능 모델) |
| :--- | :--- | :--- |
| **성능** | 적당함, 범용적 | 매우 강력함, 전문적 |
| **사용처** | 일상적인 대화, 가벼운 작업 | 복잡한 분석, 전문적인 창작 |

**실습 (5)**
- `thinking_level` 설정에 따른 추론 깊이 차이를 비교합니다.
- 같은 수학 문제를 `low`와 `high`로 풀어보고, 소요 시간과 토큰 사용량을 비교합니다.

In [21]:
# Thinking 설정 비교: thinking_level에 따른 응답 차이 확인
import time

thinking_agent = Agent(
    model_id,
    system_prompt="수학 문제를 단계별로 풀어주는 전문가입니다."
)

math_question = "한 농부가 닭과 토끼를 합쳐 20마리를 키우는데, 다리 수의 합이 56개입니다. 닭과 토끼는 각각 몇 마리인가요?"

# thinking_level='low' => 빠르지만 간단한 추론
settings_low = GoogleModelSettings(
    google_thinking_config={'thinking_level': 'low'},
    temperature=0.3,
)

# thinking_level='high' => 느리지만 깊은 추론
settings_high = GoogleModelSettings(
    google_thinking_config={'thinking_level': 'high'},
    temperature=0.3,
)

In [22]:
# low 모드 실행
start = time.time()
result_low = await thinking_agent.run(math_question, model_settings=settings_low)
time_low = time.time() - start

print(f"질문: {math_question}")
print("=" * 80)
print(f"[thinking_level='low'] (소요시간: {time_low:.2f}초)")
print("-" * 40)
print(result_low.output)

질문: 한 농부가 닭과 토끼를 합쳐 20마리를 키우는데, 다리 수의 합이 56개입니다. 닭과 토끼는 각각 몇 마리인가요?
[thinking_level='low'] (소요시간: 4.81초)
----------------------------------------
이 문제를 단계별로 아주 쉽게 풀어드리겠습니다.

### 문제 분석
*   **전체 동물의 수:** 20마리
*   **전체 다리의 수:** 56개
*   **닭의 다리 수:** 2개
*   **토끼의 다리 수:** 4개

---

### 풀이 방법 1: 가정법 (가장 쉬운 방법)

만약 20마리가 모두 '닭'이라고 가정해 봅시다.

1.  **모두 닭이라고 가정할 때의 다리 수:**
    *   20마리 × 2개 = 40개
2.  **실제 다리 수와의 차이:**
    *   56개(실제) - 40개(가정) = 16개
    *   이 16개의 다리 차이는 토끼를 닭으로 잘못 생각했기 때문에 생깁니다.
3.  **토끼 한 마리당 다리 차이:**
    *   토끼 다리(4개) - 닭 다리(2개) = 2개
    *   즉, 닭 한 마리를 토끼로 바꿀 때마다 다리가 2개씩 늘어납니다.
4.  **토끼의 수 구하기:**
    *   16개 ÷ 2개 = **8마리 (토끼)**
5.  **닭의 수 구하기:**
    *   20마리 - 8마리 = **12마리 (닭)**

---

### 풀이 방법 2: 연립방정식 (수학적 방법)

닭의 수를 $x$, 토끼의 수를 $y$라고 하면 다음과 같은 식을 세울 수 있습니다.

1.  **동물 수에 관한 식:** $x + y = 20$
2.  **다리 수에 관한 식:** $2x + 4y = 56$

**계산 과정:**
*   첫 번째 식에서 $x = 20 - y$를 얻습니다.
*   이것을 두 번째 식에 대입합니다:
    *   $2(20 - y) + 4y = 56$
    *   $40 - 2y + 4y = 56$
    *   $40 + 2y = 56$
    

In [23]:
# high 모드 실행
start = time.time()
result_high = await thinking_agent.run(math_question, model_settings=settings_high)
time_high = time.time() - start

print(f"[thinking_level='high'] (소요시간: {time_high:.2f}초)")
print("-" * 40)
print(result_high.output)

[thinking_level='high'] (소요시간: 9.54초)
----------------------------------------
안녕하세요! 수학 문제를 단계별로 명쾌하게 풀어드리겠습니다.

이 문제는 **'가정법(논리적 추론)'**을 사용하면 아주 쉽게 풀 수 있습니다. 두 가지 방법으로 설명해 드릴게요.

---

### 방법 1: 가정법 (논리적 추론)

가장 직관적인 방법입니다. 모든 동물이 닭이라고 가정하고 시작해 봅시다.

**1단계: 모두 닭이라고 가정하기**
만약 20마리가 모두 닭이라면, 다리의 수는 총 몇 개일까요?
*   20마리 × 2개(닭의 다리) = **40개**

**2단계: 실제 다리 수와 비교하기**
문제에서 주어진 다리 수는 56개입니다. 우리가 가정한 40개와는 차이가 있죠?
*   56개 - 40개 = **16개** (이만큼의 다리가 부족합니다.)

**3단계: 토끼의 수 구하기**
토끼는 닭보다 다리가 2개 더 많습니다(4개 - 2개 = 2개). 즉, 닭을 토끼로 바꿀 때마다 다리가 2개씩 늘어납니다. 부족한 다리 16개를 2로 나누면 토끼의 마릿수가 나옵니다.
*   16 ÷ 2 = **8마리 (토끼)**

**4단계: 닭의 수 구하기**
전체 20마리 중에서 토끼 8마리를 빼면 닭의 수가 나옵니다.
*   20마리 - 8마리 = **12마리 (닭)**

---

### 방법 2: 연립방정식 (수학적 풀이)

조금 더 수학적인 접근을 원하신다면 방정식을 사용할 수 있습니다.

1.  닭의 수를 $x$, 토끼의 수를 $y$라고 합니다.
2.  **식 1 (마릿수):** $x + y = 20$
3.  **식 2 (다리 수):** $2x + 4y = 56$

**풀이 과정:**
*   첫 번째 식에서 $x = 20 - y$로 바꿉니다.
*   두 번째 식에 대입합니다: $2(20 - y) + 4y = 56$
*   괄호를 풉니다: $40 - 2y + 4y = 56$
*   정리합니다: $2y = 16$
*   따라

In [24]:
# 토큰 사용량 비교
usage_low = result_low.usage()
usage_high = result_high.usage()

print("=" * 50)
print("토큰 사용량 비교")
print("=" * 50)
print(f"{'':>15} {'low':>15} {'high':>15}")
print(f"{'입력 토큰':>15} {usage_low.input_tokens or 0:>15,} {usage_high.input_tokens or 0:>15,}")
print(f"{'출력 토큰':>15} {usage_low.output_tokens or 0:>15,} {usage_high.output_tokens or 0:>15,}")

토큰 사용량 비교
                            low            high
          입력 토큰              58              58
          출력 토큰             802           1,740


### 2. 스트리밍 응답

**일반 응답 방식 (기본)**
- AI가 답변을 **완전히 다 만든 후** 에 한 번에 보여줍니다
- 긴 답변일수록 기다리는 시간이 길어집니다

**스트리밍 응답 방식 (개선)**
- AI가 답변을 **만드는 동시에 조금씩** 보여줍니다
- 사용자가 기다림 없이 답변을 바로 읽을 수 있어 더 자연스럽습니다

PydanticAI에서는 `run_stream()` + `stream_text()` 를 사용합니다.

```python
async with agent.run_stream(question) as result:
    async for text in result.stream_text(delta=True):
        print(text, end="")
```

> **참고:** `run_stream()`은 비동기(async) 전용입니다. Jupyter 노트북에서는 `await`로 바로 실행할 수 있습니다.

**실습**
- `run_stream()`으로 AI 응답을 실시간으로 출력합니다.
- `delta=True` 옵션으로 새로 생성된 텍스트만 스트리밍합니다.

In [25]:
question = "AI가 어떻게 작동하는지 간단히 설명해주세요"
print(f"질문: {question}")
print("\n실시간 응답:")

# delta=True: 이전 텍스트를 포함하지 않고 새로 생성된 부분만 출력
async with agent.run_stream(question) as result:
    async for text in result.stream_text(delta=True):
        print(text, end="", flush=True)

print(f"\n\n사용 토큰: {result.usage()}")

질문: AI가 어떻게 작동하는지 간단히 설명해주세요

실시간 응답:
AI(인공지능)가 작동하는 원리를 아주 쉽게 설명하면 **'엄청난 양의 데이터를 학습해서 패턴을 찾아내고, 이를 바탕으로 다음에 올 내용을 예측하는 것'**이라고 할 수 있습니다.

핵심 과정을 3단계로 나누어 설명해 드릴게요.

---

### 1. 학습 (데이터 입력)
AI에게 엄청난 양의 정보를 보여줍니다.
*   **예시:** 강아지가 무엇인지 알려주기 위해 수만 장의 강아지 사진을 AI에게 보여줍니다.
*   학습 과정에서 AI는 강아지의 특징(귀 모양, 코 모양, 털의 질감 등)을 수학적인 수치로 기억하게 됩니다.

### 2. 패턴 찾기 (연결망 구축)
AI 내부에는 **'신경망(Neural Network)'**이라는 복잡한 연결 고리가 있습니다. 이는 사람의 뇌 구조를 흉내 낸 것인데, 데이터를 반복해서 보면서 어떤 특징이 강아지인지, 아닌지를 구분하는 **'나만의 기준(가중치)'**을 세웁니다.

### 3. 예측 및 답변 (결과 도출)
학습이 끝난 AI에게 새로운 사진을 보여주면, AI는 자신이 배운 패턴과 비교합니다.
*   "이 사진은 95% 확률로 강아지야!"라고 판단을 내리는 것이죠.
*   **챗GPT 같은 언어 AI**라면, 앞뒤 문맥을 통해 "다음에 올 단어로 가장 적절한 것은 무엇인가?"를 확률적으로 계산해서 문장을 만듭니다.

---

### 요약하자면?
AI는 생각하는 것처럼 보이지만, 사실은 **"엄청나게 똑똑한 통계학자"**입니다.

1.  **입력:** 수많은 데이터를 넣는다.
2.  **계산:** 통계적으로 가장 가능성이 높은 패턴을 찾는다.
3.  **출력:** 그 확률에 따라 답을 제시한다.

즉, AI는 스스로 깨달음을 얻는 것이 아니라, **이미 존재하는 데이터들 사이의 규칙을 아주 빠른 속도로 찾아내어 결과를 만들어내는 기술**이라고 이해하시면 됩니다.

사용 토큰: RunUsage(input_tokens=11, output_tokens=47

### 3. 멀티턴 대화 (Multi-turn Conversation)

**멀티턴 대화란?**
- **Single-turn**: 한 번의 질문과 한 번의 응답으로 완결. 이전 대화 맥락을 기억하지 않음
- **Multi-turn**: 여러 차례 주고받으며 **이전 대화 맥락을 유지** 하는 대화 방식
- 예: "데이터 분석 취업 준비 방법" => "그중 먼저 할 것은?" => AI가 앞 대화를 기억하고 답변

PydanticAI에서 멀티턴 대화는 `message_history` 매개변수로 구현합니다.

**원리:**
1. 첫 번째 질문의 결과에서 `result.new_messages()` 로 대화 기록을 가져옵니다
2. 다음 질문에 `message_history=이전_메시지` 를 전달합니다
3. AI가 이전 대화 맥락을 기억한 상태로 응답합니다

```python
# 리스트에 누적 (대화가 길어질 때 편리)
history = []
result1 = await agent.run('첫 번째 질문', message_history=history)
history.extend(result1.new_messages())
result2 = await agent.run('두 번째 질문', message_history=history)
history.extend(result2.new_messages())
```

> **참고:** `new_messages()`는 해당 턴에서 새로 생긴 메시지만 반환합니다. 대화 기록을 직접 관리하므로 중간 수정, 분기, 저장/복원이 자유롭습니다.

**실습 (1)**
- 3턴 대화를 통해 `message_history`로 맥락을 유지하는 방법을 익힙니다.
- 이전 대화 기록을 합쳐서 전달하면 AI가 앞 대화를 기억합니다.

In [ ]:
# 시스템 프롬프트가 포함된 전문가 Agent 생성
chat_agent = Agent(
    model_id,
    system_prompt="너는 데이터 분석 분야의 시니어 전문가야. 실무 경험을 바탕으로 구체적이고 실용적인 조언을 제공해."
)

settings = GoogleModelSettings(temperature=0.8, max_tokens=500)

# 첫 번째 메시지
print("[첫 번째 대화]")
user_msg1 = "저는 지금 데이터 분석 직무에 취업을 하려해요."
print(f"사용자: {user_msg1}")

result1 = await chat_agent.run(user_msg1, model_settings=settings)
print(f"AI 전문가: {result1.output}")

# 두 번째 메시지 — message_history로 이전 대화 맥락 전달
print(f"\n[두 번째 대화]")
user_msg2 = "제가 가장 먼저 무엇을 준비하면 좋을까요?"
print(f"사용자: {user_msg2}")

result2 = await chat_agent.run(
    user_msg2,
    message_history=result1.new_messages(),  # 이전 대화 기록 전달!
    model_settings=settings
)
print(f"AI 전문가: {result2.output}")

# 세 번째 메시지 — 누적된 대화 기록 전달
print(f"\n[세 번째 대화]")
user_msg3 = "Python은 어느 정도 수준까지 알아야 하나요?"
print(f"사용자: {user_msg3}")

# 이전 두 대화의 기록을 합쳐서 전달
all_messages = result1.new_messages() + result2.new_messages()
result3 = await chat_agent.run(
    user_msg3,
    message_history=all_messages, 
    #context window 크기! 이게 크면 이전 대화 혹은 많은 양의 텍스트를 한꺼번에 처리할 수 있다!
    model_settings=settings
)
print(f"AI 전문가: {result3.output}")

[첫 번째 대화]
사용자: 저는 지금 데이터 분석 직무에 취업을 하려해요.
AI 전문가: 데이터 분석가로의 첫발을 내딛는 것을 진심으로 응원합니다. 실무에서 10년 넘게 데이터 팀을 운영하고 채용을 진행해본 경험을 바탕으로, **'취업 시장에서 정말로 원하는 데이터 분석가'**가 되기 위한 핵심 전략을 정리해 드릴게요.

---

### 1. "툴(Tool) 전문가"가 아니라 "문제 해결사"가 되어야 합니다.
많은 주니어들이 "Python, SQL, Tableau 다 할 줄 알아요"라고 어필합니다. 하지만 실무진은 **"이 기술을 써서 어떤 비즈니스 문제를 해결해봤는가?"**를 봅니다.

*   **Action:** 단순히 코드를 짜는 것에서 멈추지 마세요.
    *   **Bad:** "이커머스 데이터를 가지고 매출을 분석했다."
    *   **Good:** "재구매율이 3개월간 하락하는 문제를 발견하고, 이탈 고객의 결제 패턴을 SQL로 추출했다. 이를 통해 타겟 마케팅을 제안하여 재구매율을 5% 개선하는 가설을 세웠다."
*   **핵심:** **[데이터 처리] → [인사이트 도출] → [비즈니스 액션 제안]**의 프로세스를 프로젝트 경험에 녹여내야 합니다.

### 2. SQL은 선택이 아닌 '생존'입니다.
데이터 분석가 면접에서 가장 많이 탈락하는 이유 중 하나가 SQL 역량 부족입니다. 실무에서는 분석 모델링보다 **데이터를 추출하고 정제하는 시간**이 80% 이상입니다.

*   **Action:** 
    *   `JOIN`, `GROUP BY`, `Window Function`은 기본입니다.
    *   실제 기업의 로그 데이터와 유사한 복잡한 구조의 데이터를 다뤄보세요. (프로그래머스 SQL 고득점 Kit, LeetCode 등을 추천합니다.)
    *   **성능 고려:** 단순히 결과만 뽑는 게 아니라, 어떻게 하면 효율적으로 쿼리를 짤 수 있는지 고민하는 모습을 보여주세요.

### 3. '도메인 지식'에 대한 갈증을 보여주세요

**실습 (2)**
- 스트리밍 + 멀티턴을 결합한 대화형 채팅 루프를 실행합니다.
- `input()`으로 사용자 입력을 받고, 스트리밍으로 실시간 응답합니다.

**주의:** 이 셀은 `input()`을 사용하므로 **Run All로 전체 실행하면 여기서 멈춥니다.** 반드시 이 셀만 개별 실행하세요.

In [29]:
# 실시간 스트리밍 + 멀티턴 대화 루프
# '종료' 또는 'quit' 입력 시 대화 종료

stream_chat_agent = None

print("실시간 PydanticAI 채팅 (스트리밍)")
print("=" * 60)
print("명령어: '종료' 또는 'quit' => 채팅 종료")
print("=" * 60)

message_history = []  # 대화 기록 누적
turn_count = 0

while True:
    print(f"\n현재 대화 턴 {turn_count+1}")
    print("=" * 60)
    user_msg = input("사용자 >").strip()

    # 대화 종료
    if user_msg.lower() in ['q', 'quit', 'exit', '종료'] :
        print("\n채팅을 종료합니다!")
        break

    # user메시지가 비어있는 경우
    if not user_msg :
        print("메시지를 입력해주세요!")
        continue

    # 사용자 질문 출력
    print("사용자:", user_msg)

    # 스트리밍 응답
    print('AI:', end='', flush=True)

    async with agent.run_stream(user_msg, 
                                message_history=message_history,
                                model_settings=settings) as result:
        async for text in result.stream_text(delta=True):
            print(text, end="", flush=True)
    print() # 줄바꿈

    # 대화 기록 누적 (다음 턴에 맥락 전달)
    message_history.extend(result.new_messages())
    turn_count += 1

실시간 PydanticAI 채팅 (스트리밍)
명령어: '종료' 또는 'quit' => 채팅 종료

현재 대화 턴 1
사용자: 안녕하세요
AI:안녕하세요! 무엇을 도와드릴까요?

현재 대화 턴 2
사용자: ㄴㅇㄴㅇ
AI:'ㄴㅇㄴㅇ'은 혹시 '내일'을 줄여서 쓰신 건가요, 아니면 다른 의미가 있는 단어인가요? 

혹시 궁금한 점이 있으시거나 제가 도와드릴 일이 있다면 편하게 말씀해 주세요!

현재 대화 턴 3

채팅을 종료합니다!


# 3시 5분에 다시 시작하겠습니다~!

### 4. 도구 (Tool)

- [도구(Tool) 공식 문서](https://ai.pydantic.dev/tools/)

Agent에게 **Python 함수를 도구로 등록** 하면, AI가 필요할 때 직접 함수를 호출하여 정보를 가져옵니다.

**동작 흐름:**

![도구(Tool) 동작 흐름](images/tool_flow.png)

1. 사용자가 질문을 보냅니다
2. AI가 질문을 분석하여 **어떤 도구가 필요한지 자동으로 판단** 합니다
3. 선택된 도구(Python 함수)를 **AI가 직접 호출** 합니다
4. 도구 함수의 반환값을 받아 **최종 응답을 생성** 합니다

**실습 (1)**
- 날짜 조회, 외부 API(Wikipedia, YouTube)를 도구로 연결하여 실시간 정보를 검색합니다.
- **사전 준비:** `.env` 파일에 `YOUTUBE_API_KEY`를 설정해야 YouTube 검색이 동작합니다.

In [ ]:
import wikipediaapi
from googleapiclient.discovery import build
from datetime import date 

# 위키피디아 클라이언트 초기화 (한국어)
wiki = wikipediaapi.Wikipedia(
    user_agent='Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36',
    language='ko'
)

# YouTube Data API 클라이언트 초기화 (.env의 YOUTUBE_API_KEY 사용)
youtube = build('youtube', 'v3', developerKey=os.getenv('YOUTUBE_API_KEY'))

# 외부 API를 활용하는 도구가 포함된 Agent
search_agent = Agent(
    model_id,
    system_prompt=(
        "당신은 정보 검색 어시스턴트입니다. "
        "위키피디아와 YouTube 도구를 활용하여 정확한 정보를 제공하세요."
    ),
)

# 데코레이터 : 
# - 함수나 클래스를 감싸서 기존 함수의 기능을 그대로 유지하면서 
# - 새로운 함수를 만들고 싶은 경우
# - 실행 방식이나 등록 방법을 바꿔 주는 역할

@search_agent.tool_plain
def get_current_date() -> str:
    """오늘 날짜를 반환합니다."""
    return str(date.today())

# def wrapp_deco() :
#     print("날짜 함수를 호출할겁니다!")
#     get_current_date()
#     print("날짜 함수를 호출이 완료되었습니다")

# 그럼 함수 시그니처+타입 힌트랑 독스트링을 통해 llm이 판단을 하는거네요..?
# 함수 이름 + 함수 파라미터 + 독 스트링을 기반으로 LLM이 도구를 이해를 한다!
@search_agent.tool_plain
def search_wikipedia(query: str) -> str:
    # 독스트링 : docstring
    """
    위키피디아에서 검색어에 해당하는 문서의 요약을 반환합니다.

    args : 
        - query : 검색어 ex) 방탄소년단, 인공지능
    
    return :
        - 검색어 따른 응답 결과 
        - 응답형식 : title + 요약 + url
    """
    page = wiki.page(query)
    if page.exists():
        summary = page.summary[:500]
        url = page.fullurl
        return f"[{page.title}]\n{summary}\n\n참조: {url}"
    return f"'{query}'에 대한 위키피디아 문서를 찾을 수 없습니다."


@search_agent.tool_plain
def search_youtube(query: str, max_results: int = 3) -> str:
    """YouTube에서 영상을 검색하여 제목과 링크를 반환합니다."""
    try:
        response = youtube.search().list(
            q=query,
            part='snippet',
            type='video',
            maxResults=max_results
        ).execute()

        results = []
        for item in response.get('items', []):
            title = item['snippet']['title']
            video_id = item['id']['videoId']
            url = f"https://www.youtube.com/watch?v={video_id}"
            results.append(f"- {title}\n  {url}")
        return "\n".join(results) if results else "검색 결과가 없습니다."
    except Exception as e:
        return f"YouTube 검색 오류: {e}"


# 외부 API 도구를 활용하는 질문들
questions = [
    "오늘 날짜와 더불어 위키피디아에서 '인공지능'을 검색해주세요.", #도구 몇개??
    # "YouTube에서 '파이썬 데이터 분석 입문' 영상을 찾아주세요.",
]

for q in questions:
    result = await search_agent.run(q)
    print(f"Q: {q}")
    print(f"A: {result.output}")
    print("-" * 50)

Q: 오늘 날짜와 더불어 위키피디아에서 '인공지능'을 검색해주세요.
A: 오늘 날짜는 **2026년 4월 2일**입니다. 요청하신 '인공지능'에 대한 위키피디아 검색 결과는 다음과 같습니다.

**[인공지능(Artificial Intelligence, AI)]**
인공지능은 인간의 학습 능력, 추론 능력, 지각 능력을 컴퓨터 과학을 통해 인공적으로 구현하는 기술입니다. 이는 단순히 인간의 지능을 모방하는 시스템을 넘어, 그러한 지능을 생성하기 위한 방법론과 실현 가능성을 연구하는 과학 기술 분야를 포괄합니다. 인간의 '자연 지능'과는 구분되는 개념으로, 현대 정보공학의 핵심 인프라 기술로 자리 잡고 있습니다.

더 자세한 내용은 [위키피디아 인공지능 문서](https://ko.wikipedia.org/wiki/%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5)에서 확인하실 수 있습니다.
--------------------------------------------------


**실습 (2)**
- 어떤 도구가 호출되었는지 `result.new_messages()`로 확인합니다.
- AI가 도구를 호출한 전체 과정(요청 => 도구 호출 => 응답)을 살펴봅니다.

In [ ]:
# # 어떤 툴이 호출되었는지 확인
result.new_messages()

# [ModelRequest(parts=[SystemPromptPart(content='당신은 정보 검색 어시스턴트입니다. 위키피디아와 YouTube 도구를 활용하여 정확한 정보를 제공하세요.', timestamp=datetime.datetime(2026, 4, 2, 6, 15, 55, 784126, tzinfo=datetime.timezone.utc)), UserPromptPart(content="YouTube에서 '파이썬 데이터 분석 입문' 영상을 찾아주세요.", timestamp=datetime.datetime(2026, 4, 2, 6, 15, 55, 784126, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 2, 6, 15, 55, 785123, tzinfo=datetime.timezone.utc), run_id='d27a7640-1863-4128-87bd-be8945636ed7'),
#  LLM판단 -> 도구실행: ModelResponse(parts=[ToolCallPart(tool_name='search_youtube', args={'query': '파이썬 데이터 분석 입문'}, tool_call_id='vsdz087z', provider_name='google-gla', provider_details={'thought_signature': 'EjQKMgG+Pvb7+ujzuSnOvqXAEkpjraaRRDlKZd9V6q2jkevPd9EkY1oPTJIaPD+O72WJ/TnT'})], usage=RequestUsage(input_tokens=197, output_tokens=22, details={'text_prompt_tokens': 197}), model_name='gemini-3.1-flash-lite-preview', timestamp=datetime.datetime(2026, 4, 2, 6, 15, 56, 826756, tzinfo=datetime.timezone.utc), provider_name='google-gla', provider_url='https://generativelanguage.googleapis.com/', provider_details={'finish_reason': 'STOP'}, provider_response_id='nQnOacSqFaGW0-kP362G-As', finish_reason='stop', run_id='d27a7640-1863-4128-87bd-be8945636ed7'),
#  도구호출결과 : ModelRequest(parts=[ToolReturnPart(tool_name='search_youtube', content='- 파이썬 코딩 무료 강의 (활용편5) - 데이터 분석 및 시각화, 이 영상 하나로 끝내세요\n  https://www.youtube.com/watch?v=PjhlUzp_cU0\n- [기초파이썬] 1강.코랩 시작하기 | 데이터분석 기초 | 파이썬 기초 | 빅분기 실기\n  https://www.youtube.com/watch?v=EYKagoL2J0U\n- 😎동네 마트에 컴공과가 취직하면..? (#파이썬 #데이터분석)\n  https://www.youtube.com/watch?v=ttP3zt4jL2Y', tool_call_id='vsdz087z', timestamp=datetime.datetime(2026, 4, 2, 6, 15, 57, 291320, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 2, 6, 15, 57, 291320, tzinfo=datetime.timezone.utc), run_id='d27a7640-1863-4128-87bd-be8945636ed7'),
#  LLM의 최종응답 ModelResponse(parts=[TextPart(content="'파이썬 데이터 분석 입문'과 관련된 YouTube 영상들을 찾아보았습니다. 학습에 도움이 될 만한 영상들을 추천해 드립니다.\n\n1. **파이썬 코딩 무료 강의 (활용편5) - 데이터 분석 및 시각화, 이 영상 하나로 끝내세요**\n   - 링크: https://www.youtube.com/watch?v=PjhlUzp_cU0\n\n2. **[기초파이썬] 1강.코랩 시작하기 | 데이터분석 기초 | 파이썬 기초 | 빅분기 실기**\n   - 링크: https://www.youtube.com/watch?v=EYKagoL2J0U\n\n3. **😎동네 마트에 컴공과가 취직하면..? (#파이썬 #데이터분석)**\n   - 링크: https://www.youtube.com/watch?v=ttP3zt4jL2Y\n\n위 영상들을 통해 파이썬 데이터 분석의 기초를 학습해 보시길 바랍니다!", provider_name='google-gla', provider_details={'thought_signature': 'EjQKMgG+Pvb7/GnYu2cGDmzyVR+aPXtZchwLIZ+j8CuJQ3sG8NZgPAOIu9frxfuykTCY5HVD'})], usage=RequestUsage(input_tokens=387, output_tokens=222, details={'text_prompt_tokens': 387}), model_name='gemini-3.1-flash-lite-preview', timestamp=datetime.datetime(2026, 4, 2, 6, 15, 59, 913127, tzinfo=datetime.timezone.utc), provider_name='google-gla', provider_url='https://generativelanguage.googleapis.com/', provider_details={'finish_reason': 'STOP'}, provider_response_id='oAnOaf-QGdOZ0-kPtMTrmQw', finish_reason='stop', run_id='d27a7640-1863-4128-87bd-be8945636ed7')]


# Google Gemini모델쓴다!
# Gemini 내부모델 : 단일 LLM모델 + tool을 호출하고 잘 활용할 수 있도록 for문 같은 시스템이 있음!


# [ModelRequest(parts=[SystemPromptPart(content='당신은 정보 검색 어시스턴트입니다. 위키피디아와 YouTube 도구를 활용하여 정확한 정보를 제공하세요.', timestamp=datetime.datetime(2026, 4, 2, 6, 24, 47, 732092, tzinfo=datetime.timezone.utc)), UserPromptPart(content="오늘 날짜와 더불어 위키피디아에서 '인공지능'을 검색해주세요.", timestamp=datetime.datetime(2026, 4, 2, 6, 24, 47, 732092, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 2, 6, 24, 47, 732092, tzinfo=datetime.timezone.utc), run_id='40c899de-24b1-427c-a647-862b973140f0'),
#  ModelResponse(parts=[ToolCallPart(tool_name='get_current_date', args={}, tool_call_id='4tl52rze', provider_name='google-gla', provider_details={'thought_signature': 'EjQKMgG+Pvb72F6Iw7FsGNaC7wx9sTJIcAnxm9CQ+slltr/fWq4R3dzBhaD5X3yoieaqGP+N'}), ToolCallPart(tool_name='search_wikipedia', args={'query': '인공지능'}, tool_call_id='8hef6iqq')], usage=RequestUsage(input_tokens=203, output_tokens=31, details={'text_prompt_tokens': 203}), model_name='gemini-3.1-flash-lite-preview', timestamp=datetime.datetime(2026, 4, 2, 6, 24, 48, 982903, tzinfo=datetime.timezone.utc), provider_name='google-gla', provider_url='https://generativelanguage.googleapis.com/', provider_details={'finish_reason': 'STOP'}, provider_response_id='sQvOad3hHq6q0-kPipnu0Ao', finish_reason='stop', run_id='40c899de-24b1-427c-a647-862b973140f0'),
#  ModelRequest(parts=[ToolReturnPart(tool_name='get_current_date', content='2026-04-02', tool_call_id='4tl52rze', timestamp=datetime.datetime(2026, 4, 2, 6, 24, 48, 985806, tzinfo=datetime.timezone.utc)), ToolReturnPart(tool_name='search_wikipedia', content='[인공지능]\n인공지능(人工智能, 영어: artificial intelligence, AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부분야 중 하나이다. 정보공학 분야에 있어 하나의 인프라 기술이기도 하다. 인간을 포함한 동물이 갖고 있는 지능 즉, 자연 지능(natural intelligence)과는 다른 개념이다.\n인간의 지능을 모방한 기능을 갖춘 컴퓨터 시스템이며, 인간의 지능을 기계 등에 인공적으로 시연(구현)한 것이다. 일반적으로 범용 컴퓨터에 적용한다고 가정한다. 이 용어는 또한 그와 같은 지능을 만들 수 있는 방법론이나 실현 가능성 등을 연구하는 과학 기술 분야를 지칭하기도 한다.\n\n참조: https://ko.wikipedia.org/wiki/%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5', tool_call_id='8hef6iqq', timestamp=datetime.datetime(2026, 4, 2, 6, 24, 50, 816496, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 2, 6, 24, 50, 817496, tzinfo=datetime.timezone.utc), run_id='40c899de-24b1-427c-a647-862b973140f0'),
#  ModelResponse(parts=[TextPart(content="오늘 날짜는 **2026년 4월 2일**입니다. 요청하신 '인공지능'에 대한 위키피디아 검색 결과는 다음과 같습니다.\n\n**[인공지능(Artificial Intelligence, AI)]**\n인공지능은 인간의 학습 능력, 추론 능력, 지각 능력을 인공적으로 구현하려는 컴퓨터 과학의 세부 분야입니다. 정보공학 분야의 인프라 기술로서, 인간의 지능을 모방한 기능을 갖춘 컴퓨터 시스템을 의미합니다. 또한, 이러한 지능을 기계 등에 구현하는 방법론이나 실현 가능성을 연구하는 과학 기술 분야를 통칭하기도 합니다.\n\n더 자세한 내용은 위키피디아에서 확인하실 수 있습니다: [https://ko.wikipedia.org/wiki/%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5](https://ko.wikipedia.org/wiki/%EC%9D%B8%EA%B3%B5%EC%A7%80%EB%8A%A5)", provider_name='google-gla', provider_details={'thought_signature': 'EjQKMgG+Pvb7ZbVpAIjbwbaZwjZwFcN5Ips8ll+nm3VuyJg3pnnAUIWxXCBwwlEpU3O6dwp1'})], usage=RequestUsage(input_tokens=491, output_tokens=245, details={'text_prompt_tokens': 491}), model_name='gemini-3.1-flash-lite-preview', timestamp=datetime.datetime(2026, 4, 2, 6, 24, 53, 6673, tzinfo=datetime.timezone.utc), provider_name='google-gla', provider_url='https://generativelanguage.googleapis.com/', provider_details={'finish_reason': 'STOP'}, provider_response_id='tQvOaa2qHo6e0-kP4cD1kQw', finish_reason='stop', run_id='40c899de-24b1-427c-a647-862b973140f0')]


[ModelRequest(parts=[SystemPromptPart(content='당신은 정보 검색 어시스턴트입니다. 위키피디아와 YouTube 도구를 활용하여 정확한 정보를 제공하세요.', timestamp=datetime.datetime(2026, 4, 2, 6, 24, 47, 732092, tzinfo=datetime.timezone.utc)), UserPromptPart(content="오늘 날짜와 더불어 위키피디아에서 '인공지능'을 검색해주세요.", timestamp=datetime.datetime(2026, 4, 2, 6, 24, 47, 732092, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 4, 2, 6, 24, 47, 732092, tzinfo=datetime.timezone.utc), run_id='40c899de-24b1-427c-a647-862b973140f0'),
 ModelResponse(parts=[ToolCallPart(tool_name='get_current_date', args={}, tool_call_id='4tl52rze', provider_name='google-gla', provider_details={'thought_signature': 'EjQKMgG+Pvb72F6Iw7FsGNaC7wx9sTJIcAnxm9CQ+slltr/fWq4R3dzBhaD5X3yoieaqGP+N'}), ToolCallPart(tool_name='search_wikipedia', args={'query': '인공지능'}, tool_call_id='8hef6iqq')], usage=RequestUsage(input_tokens=203, output_tokens=31, details={'text_prompt_tokens': 203}), model_name='gemini-3.1-flash-lite-preview', timestamp=datetime.datetime(2026, 4

### 5. 비동기 동시 호출 (Concurrent Requests)

**(1) 동기(Synchronous) vs 비동기(Asynchronous) 개념**

| | 동기 (Synchronous) | 비동기 (Asynchronous) |
|---|---|---|
| **동작 방식** | 작업이 끝날 때까지 기다렸다가 다음 작업 실행 | 작업을 보내놓고 다른 작업을 먼저 진행 |
| **장점** | 코드가 단순하고 직관적 | 여러 작업을 동시에 처리하여 시간 절약 |
| **단점** | 여러 작업을 순차적으로만 처리 (느림) | 코드가 조금 더 복잡 (`async`/`await` 필요) |

**Python 비동기 핵심 문법:**

| 문법 | 역할 |
|------|------|
| `async def` | 비동기 함수 (코루틴) 정의 |
| `await` | 비동기 작업이 완료될 때까지 대기 |
| `asyncio.gather()` | 여러 비동기 작업을 동시에 실행 |

**(2) `asyncio.gather`를 활용한 동시 호출**

```python
# 동기 방식: 3개 요청 => 순차 실행 => 총 9초
result1 = agent.run_sync('질문1')  # 3초
result2 = agent.run_sync('질문2')  # 3초
result3 = agent.run_sync('질문3')  # 3초

# 비동기 동시 호출: 3개 요청 => 동시 실행 => 총 3초
result1, result2, result3 = await asyncio.gather(
    agent.run('질문1'),
    agent.run('질문2'),
    agent.run('질문3'),
)
```

> **참고:** `run_sync()`는 동시 호출이 불가능합니다. 비동기 `run()`만 `asyncio.gather`와 함께 사용할 수 있습니다.

**실습 (1)**
- `asyncio.gather()`로 3개 질문을 동시에 호출하는 기본 예시입니다.
- 한 줄로 여러 요청을 동시에 보내고, 결과를 리스트로 받습니다.

In [ ]:
import asyncio
import time

concurrent_agent = Agent(
    model_id,
    system_prompt="질문에 3문장 이내로 간결하게 답하세요."
)
settings = GoogleModelSettings(temperature=0.7, max_tokens=200)

start = time.time()
concurrent_results = await asyncio.gather(
    concurrent_agent.run("1+1?", model_settings=settings), 
     concurrent_agent.run("2+2?", model_settings=settings),
     concurrent_agent.run("3+3?", model_settings=settings)
)

pprint(concurrent_results)

# 무료 API를 쓰고 있음
# 1분에 호출할 수 있는 횟수가 정해져있음
# 하루동안 무료로 호출할 수 있는 횟수가 정해져있음

[AgentRunResult(output='1 더하기 1은 2입니다.'),
 AgentRunResult(output='2 더하기 2는 4입니다.'),
 AgentRunResult(output='3 더하기 3은 6입니다.')]


**실습 (2)**
- 순차 실행과 동시 실행의 속도 차이를 직접 측정하여 비교합니다.
- 5개 질문을 하나씩 처리하는 것과 `asyncio.gather()`로 한꺼번에 처리하는 것의 시간 차이를 확인합니다.

In [ ]:
# 비교를 위한 Agent 생성
concurrent_agent = Agent(
    model_id,
    system_prompt="질문에 3문장 이내로 간결하게 답하세요."
)

questions = [
    "Python의 장점은?",
    "데이터 분석에서 SQL이 중요한 이유는?",
    "머신러닝과 딥러닝의 차이점은?",
    "판다스(Pandas) 라이브러리란?",
    "A/B 테스트란 무엇인가?",
]

settings = GoogleModelSettings(temperature=0.7, max_tokens=200)

In [37]:
# 순차 실행 (하나씩 기다리며 실행)
print("[순차 실행] 5개 요청을 하나씩 처리")
print("=" * 60)

start = time.time()
sequential_results = []
for q in questions:
    r = await concurrent_agent.run(q, model_settings=settings)
    sequential_results.append(r)
seq_time = time.time() - start

for q, r in zip(questions, sequential_results):
    print(f"Q: {q}")
    print(f"A: {r.output[:80]}...")
    print()

print(f"순차 실행 소요 시간: {seq_time:.2f}초")

[순차 실행] 5개 요청을 하나씩 처리
Q: 오늘 날짜와 더불어 위키피디아에서 '인공지능'을 검색해주세요.
A: 오늘은 2024년 5월 22일 수요일입니다. 위키피디아에 따르면 인공지능(AI)은 인간의 학습 능력, 추론 능력, 지각 능력을 인공적으로 구현한...

순차 실행 소요 시간: 17.10초


In [38]:
# 동시 실행 (asyncio.gather로 병렬 처리)
print("[동시 실행] 5개 요청을 한꺼번에 처리")
print("=" * 60)

start = time.time()
concurrent_results = await asyncio.gather(
    *[concurrent_agent.run(q, model_settings=settings) for q in questions]
)
con_time = time.time() - start

for q, r in zip(questions, concurrent_results):
    print(f"Q: {q}")
    print(f"A: {r.output[:80]}...")
    print()

print(f"동시 실행 소요 시간: {con_time:.2f}초")

[동시 실행] 5개 요청을 한꺼번에 처리
Q: 오늘 날짜와 더불어 위키피디아에서 '인공지능'을 검색해주세요.
A: 오늘은 2024년 5월 22일 수요일입니다. 위키피디아에 따르면 인공지능(AI)은 인간의 학습 능력, 추론 능력, 지각 능력을 인공적으로 구현한...

동시 실행 소요 시간: 1.18초


In [39]:
# 결과 비교
print("=" * 50)
print("[결과 비교]")
print("=" * 50)
print(f"  순차 실행: {seq_time:.2f}초")
print(f"  동시 실행: {con_time:.2f}초")
if seq_time > con_time:
    print(f"  시간 절약: {seq_time - con_time:.2f}초 ({(1 - con_time/seq_time)*100:.0f}% 단축)")

[결과 비교]
  순차 실행: 17.10초
  동시 실행: 1.18초
  시간 절약: 15.92초 (93% 단축)


# 데이터 전처리 끝나야
# EDA


# 순차적으로 해야할 업무
# 독립적인 업무는 병렬처리 가능
# 협업! <-- 팀 협업할 때

주은님 작업이 다끝나고 -> 희진님이 일을 해야하는 경우
통계작업 ==> PPT를 만든다!

머신러닝 실험 (병렬!)
피카츄 - 로지스틱회귀 실험
치코리타 - 랜덤포레스트 실험


# 멀티 프로세싱 (상위 개념)
# - 비동기  
# - 멀티 스레딩
------
프로그램의 흐름 - 쓰레드!!!
- 메인 쓰레드 
  - 세부 스레드 (함수1 호출)
- 메인 쓰레드 흐름

비동기 처리
- 메인 스레드 1개
 - 비동기 함수 1 
 - 비동기 함수 2
- 충돌 위험도 다소 없고 메모리공유

 각 함수를 번갈아가면서 실행! ==> 동시실행! 비동기 메인 스레드안에서 작업이되서 메모리를 서로다 공유!!

# 멀티 스레딩 
- 메인 스레드 1
 - 서브 스레드 1 (작업 A를 수행) 메인스레드와 독립적인환경
 - 서브 스레드 2 (작업 B를 수행) 메인스레드와 독립적인환경 
 - 충돌날 위험도 있고!